# [1장 3강] - 학습 파이프라인 순서 이해 실습

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
# PyTorch의 난수 생성기 고정 (가중치 초기화, dropout 등에 영향)
torch.manual_seed(SEED)
# NumPy의 난수 생성기 고정 (데이터 셔플, 무작위 샘플링 등에 영향)
np.random.seed(SEED)

## 필수 1: 학습 파이프라인 순서 정렬
DataLoader부터 평가까지 학습 파이프라인 카드를 올바른 순서로 배열합니다.

In [3]:
# 필수 1. 학습 파이프라인 순서 카드 정렬
cards = ['optimizer.step()', 'loss 계산', 'DataLoader에서 batch 꺼내기', 'model(x) forward', 'loss.backward()', '평가 지표 계산']
print('섞인 카드:', cards)

# TODO: 올바른 순서로 카드 이름을 적어보세요.
ordered_cards = ["DataLoader에서 batch 꺼내기","model(x) forward", "loss 계산", "loss.backward()", "optimizer.step()", "평가 지표 계산"]
print('내가 정리한 순서:', ordered_cards if ordered_cards else 'TODO')

섞인 카드: ['optimizer.step()', 'loss 계산', 'DataLoader에서 batch 꺼내기', 'model(x) forward', 'loss.backward()', '평가 지표 계산']
내가 정리한 순서: ['DataLoader에서 batch 꺼내기', 'model(x) forward', 'loss 계산', 'loss.backward()', 'optimizer.step()', '평가 지표 계산']


## 필수 2: 한 batch 학습 흐름 주석 달기
forward, loss, zero_grad, backward, step의 역할을 코드 옆에 주석으로 적습니다.

In [4]:
# 필수 2. 한 batch 학습 흐름에 주석 달기
x = torch.randn(8, 3)
y = torch.randn(8, 1)
model = nn.Linear(3, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# TODO: 각 줄 오른쪽 주석에 단계명을 적어보세요.
pred = model(x)              # 1. 순전파: 모델에 입력값을 넣어 예측값 계산
loss = loss_fn(pred, y)      # 2. 손실 계산: 예측과 정답의 오차를 구함
optimizer.zero_grad()        # 3. 기울기 초기화: 이전 step의 gradient를 0으로 리셋 
loss.backward()              # 4. 역전파: 각 가중치의 gradient 계산
optimizer.step()             # 5. 가중치 갱신: gradient로 w를 갱신 (w = w - lr * gradient)
print('loss:', round(loss.item(), 4))

print(model.weight.grad)   # 가중치 3개의 gradient → 값 3개 나옴
print(model.bias.grad)     # bias의 gradient → 값 1개


loss: 1.5057
tensor([[ 1.1602, -0.4341,  1.6816]])
tensor([-1.9729])


왜 값이 3개가 아니라 1개인가
- loss는 "가중치별 오차"가 아니라 "모델 전체가 이번에 얼마나 틀렸나" 를 나타내는 하나의 종합 점수
- 하나의 loss를 각 가중치로 미분하면, 가중치마다 gradient가 하나씩 나온다
- 각 샘플마다 오차 -> MSELoss는 이 8개 오차를 평균내서 하나로 만든다. (MSE = Mean Squared Error)
- 각 층이 독립된 weight 텐서로 따로 존재

## 심화 1: train과 validation 구분
평가 단계에서 model.eval()과 torch.no_grad()가 들어가는 위치를 확인합니다.

In [5]:
# 심화 1. train 단계와 validation 단계 구분하기
x_valid = torch.randn(4, 3)
y_valid = torch.randn(4, 1)

# TODO: 평가 단계에서는 gradient 계산이 필요 없습니다. 아래 코드를 완성해보세요.
model.eval()
with torch.no_grad():
    valid_pred = model(x_valid)
    valid_loss = loss_fn(valid_pred, y_valid)
print('valid_loss:', round(valid_loss.item(), 4))

valid_loss: 1.6533
